# QuALITY full-dev confirmation on Colab

This notebook runs the Qwen2.5-1.5B confirmation on all 2,086 labelled QuALITY dev questions. It uses a segmented uncached reference once per workload, then compares document, fixed-block-256, radix, CPU FP16, CPU INT8/Triton, and GDSF where each comparison is informative.

The 12 runs are split into four checkpoints. On a T4, expect roughly 12--18 GPU-hours in total; individual runtimes vary. Every command is resumable at completed-run granularity. Smoke timings are functional checks only and must not be reported as evidence.

In [ ]:
import os

# Set allocator behavior before importing torch.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi

## 1. Clone or update the repository

In [ ]:
from pathlib import Path

repo = Path("/content/rag-kvcache")
!test -d /content/rag-kvcache || git clone https://github.com/i0nut02/rag-kvcache.git /content/rag-kvcache
os.chdir(repo)
!git pull --ff-only
print(Path.cwd())

In [ ]:
import torch
import triton

assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print("Torch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Optional: add HF_TOKEN as a Colab secret to avoid anonymous Hub limits.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print("HF token configured:", bool(hf_token))

## 2. Download and validate QuALITY dev

Only the labelled dev file is needed for this suite.

In [ ]:
data_dir = Path("data/quality-v1.0.1")
data_dir.mkdir(parents=True, exist_ok=True)
!wget -q -nc -P {data_dir} https://raw.githubusercontent.com/nyu-mll/quality/main/data/v1.0.1/QuALITY.v1.0.1.htmlstripped.dev
!python experiments/run_quality.py validate-data data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev --split dev --verify-counts

In [ ]:
# Focused checks for reference alignment and tensor storage. Matrix expansion below validates the new config.
!python -m unittest tests.test_reference tests.test_tensors -q
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --show-commands

## 3. Optional checkpoint restore

If a previous Colab runtime expired, upload the most recent `full-dev-checkpoint-*.zip` from your computer and call `restore_checkpoint()`. The ZIP is unpacked into the exact result directory used by `--resume`. Do not rename or edit files inside the archive.

In [ ]:
import shutil
from google.colab import files

full_results = Path("results/full_dev_confirmation/full")
full_results.mkdir(parents=True, exist_ok=True)

def restore_checkpoint():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one checkpoint ZIP")
    archive = Path(next(iter(uploaded)))
    shutil.unpack_archive(archive, full_results)
    print("Restored:", archive, "to", full_results)

def download_checkpoint(label, source=full_results):
    source = Path(source)
    archive = shutil.make_archive(f"/content/{label}", "zip", root_dir=source)
    print("Created:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MiB)")
    files.download(archive)

# Uncomment only when restoring a downloaded checkpoint in a new runtime.
restore_checkpoint()
restore_checkpoint()
restore_checkpoint()

## 4. Ten-request functional smoke

This exercises all 12 paths, including Triton, before spending hours. It writes to a separate directory and does not affect the full profile.

In [ ]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile smoke --execute --resume

In [ ]:
import json
import pandas as pd

def show_summaries(profile="full"):
    rows = []
    for path in sorted(Path(f"results/full_dev_confirmation/{profile}").glob("*.summary.json")):
        row = json.loads(path.read_text())
        row["run"] = path.name.removesuffix(".summary.json")
        rows.append(row)
    columns = [
        "run", "requests", "workload", "cache_strategy", "policy",
        "storage", "ttft_mean_s", "ttft_p95_s",
        "article_token_hit_rate", "accuracy",
        "reference_label_agreement", "reference_label_mismatches",
    ]
    return pd.DataFrame(rows).reindex(columns=columns)

show_summaries("smoke")

## 5. Full random trace, checkpoint 1/2

Runs 1--3: segmented reference, document FP16, and tuned fixed-block-256 FP16. The first run is intentionally the slowest because it computes every token without document reuse.

In [ ]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 3
show_summaries("full")

In [ ]:
download_checkpoint("full-dev-checkpoint-03")

## 6. Full random trace, checkpoint 2/2

Runs 4--6 add radix, CPU FP16 offload, and CPU INT8 with Triton restore. `--resume` skips runs 1--3. The random trace contains every dev question exactly once, so this is the suite's standard QuALITY accuracy result.

In [ ]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 6
show_summaries("full")

In [ ]:
download_checkpoint("full-dev-checkpoint-06-random-complete")

## 7. Full Zipf trace, checkpoint 1/2

Runs 7--9 create the aligned Zipf reference and compare document with fixed-block-256. Zipf repeats real questions according to article popularity; its accuracy is not the standard one-pass QuALITY accuracy.

In [ ]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 9
show_summaries("full")

In [ ]:
download_checkpoint("full-dev-checkpoint-09")

## 8. Full Zipf trace, checkpoint 2/2

Runs 10--12 add radix, document GDSF, and CPU INT8/Triton. Re-running this cell is safe: completed JSONLs with their manifests are skipped.

In [ ]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume
show_summaries("full")

## 9. Validate, analyze, and download the final evidence

The analyzer rejects misaligned traces, incompatible schemas/manifests, and missing runs. It creates paired speedups, correctness diagnostics, tables, and figures.

In [ ]:
!python experiments/run_quality.py analyze-inference results/full_dev_confirmation/full \
    --suite-config configs/full_dev_analysis.json \
    --output-dir results/full_dev_confirmation/analysis \
    --bootstrap-samples 20000 --seed 42

analysis_dir = Path("results/full_dev_confirmation/analysis")
display(pd.read_csv(analysis_dir / "run_summaries.csv"))
display(pd.read_csv(analysis_dir / "fair_speedups.csv"))
display(pd.read_csv(analysis_dir / "correctness.csv"))

In [ ]:
download_checkpoint(
    "full-dev-confirmation-complete",
    source=Path("results/full_dev_confirmation"),
)

The browser downloads do not use Google Drive quota. Verify each ZIP locally before ending a runtime. Keep the raw JSONL, summaries, and manifests together; only curated generated analysis should later be committed to Git.